In [ ]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader


import os
import torch
import torch.nn as nn
import torch.functional as F
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
from LSTM import LSTM

In [ ]:
csv_path = os.path.join('dataset', 'data_final.csv')

dataset = pd.read_csv(
    csv_path,
    index_col='Date',
    parse_dates=['Date']
)

dataset.head()

,전력량,Weekend,Vacation,time_sin,time_cos,DayOfWeek,day_sin,day_cos
Date,,,,,,,,
2021-01-01 00:15:00,62.0,0.0,0.0,0.065403,0.997859,4,-0.433884,-0.900969
2021-01-01 00:30:00,61.0,0.0,0.0,0.130526,0.991445,4,-0.433884,-0.900969
2021-01-01 00:45:00,61.0,0.0,0.0,0.195090,0.980785,4,-0.433884,-0.900969
2021-01-01 01:00:00,61.0,0.0,0.0,0.258819,0.965926,4,-0.433884,-0.900969
2021-01-01 01:15:00,96.0,0.0,0.0,0.321439,0.946930,4,-0.433884,-0.900969


In [ ]:
split1 = int(len(dataset) * 0.6)
split2 = int(len(dataset) * 0.8)

print(f'검증 셋 기준 날짜 : {dataset.index[split1]}')
print(f'테스트 셋 기준 날짜 : {dataset.index[split2]}')

In [ ]:
train = dataset.loc[:'2021-06-04 23:45']

valid = dataset.loc[
    '2021-06-05 00:00':'2021-07-25 23:45'
]

test = dataset.loc['2021-07-26 00:00':]

print(f'훈련 셋 크기 : {len(train)}, 검증 셋 크기 : {len(valid)}, 테스트 셋 크기 : {len(test)}')

In [ ]:
scaler = StandardScaler()

scaler.fit_transform(train)
scaler.transform(valid)
scaler.transform(test)

In [ ]:
def make_sequences(data, window_size = 96):
    
    X = []
    y = []

    features = data.columns
    target = '전력량'

    feature_data = data[features].values
    target_data = data[target].values

    for i in range(window_size, len(data)):
        
        X.append(
            feature_data[i-window_size:i]
        )
        
        y.append(
            target_data[i]
        )

    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = make_sequences(train, window_size=96)

X_valid, y_valid = make_sequences(valid, window_size=96)

X_test, y_test = make_sequences(test, window_size=96)

print(f'훈련 셋 feature : {X_train.shape}, target : {y_train.shape}')
print(f'검증 셋 feature : {X_valid.shape}, target : {y_valid.shape}')
print(f'테스트 셋 feature : {X_test.shape}, target : {y_test.shape}')

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_valid = torch.tensor(X_valid, dtype=torch.float32)
y_valid = torch.tensor(y_valid, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
valid_dataset = TensorDataset(X_valid, y_valid)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)